# Skin cancer image classifier

> Educational engineering example only; this is not a medical diagnosis.

The notebook expects `train/` and `test/` class folders. It validates the dataset before training, keeps the held-out test set out of model selection, and writes the model only after verifying the artifact exists.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from tensorflow.keras import Input
from tensorflow.keras.layers import Conv2D, Dense, Dropout, Flatten, MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator,
)

SUPPORTED_IMAGE_EXTENSIONS = {".bmp", ".jpeg", ".jpg", ".png", ".ppm", ".tif", ".tiff"}


In [ ]:
# All paths are configurable and remain outside source control by default.
DATA_ROOT = Path(os.environ.get("SKINCANCER_DATA_ROOT", ".")).expanduser()
train_dir = DATA_ROOT / "train"
test_dir = DATA_ROOT / "test"
model_path = Path(os.environ.get("MODEL_PATH", "artifacts/skin_cancer_classifier.keras")).expanduser()
image_size = (150, 150)
batch_size = 32
validation_split = 0.2
random_seed = 42

if not 0 < validation_split < 1:
    raise ValueError("validation_split must be strictly between 0 and 1.")
if model_path.suffix.lower() not in {".keras", ".h5"}:
    raise ValueError("MODEL_PATH must end with .keras or .h5 for Keras model saving.")

for dataset_dir in (train_dir, test_dir):
    if not dataset_dir.is_dir():
        raise FileNotFoundError(
            f"Dataset directory is missing: {dataset_dir}. "
            "Set SKINCANCER_DATA_ROOT or add train/test folders before running."
        )

def image_files(directory):
    """Return files that flow_from_directory can actually decode as image inputs."""
    return sorted(
        path
        for path in directory.rglob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS
    )

train_classes = sorted(path.name for path in train_dir.iterdir() if path.is_dir())
test_classes = sorted(path.name for path in test_dir.iterdir() if path.is_dir())
if len(train_classes) != 2:
    raise ValueError(f"Expected exactly two training class folders; found {train_classes}.")
if train_classes != test_classes:
    raise ValueError(
        f"Train/test class folders differ: train={train_classes}, test={test_classes}."
    )

train_counts = {label: len(image_files(train_dir / label)) for label in train_classes}
test_counts = {label: len(image_files(test_dir / label)) for label in test_classes}
# Keras rounds the directory split per class; five files are needed for a 20% split
# to contain at least one validation image while leaving training data available.
minimum_train_images = max(2, int(np.ceil(1 / validation_split)))
if any(count < minimum_train_images for count in train_counts.values()):
    raise ValueError(
        "Each training class must contain at least "
        f"{minimum_train_images} supported image files; found {train_counts}."
    )
if any(count < 1 for count in test_counts.values()):
    raise ValueError(
        f"Each test class must contain at least one supported image file; found {test_counts}."
    )

# Seed Python/TensorFlow operations for reproducible runs where supported by the hardware.
tf.keras.utils.set_random_seed(random_seed)
try:
    tf.config.experimental.enable_op_determinism()
except (AttributeError, RuntimeError):
    print("Deterministic TensorFlow ops are unavailable; results may vary slightly.")


In [ ]:
# Validation is drawn only from train/. test/ remains untouched until final evaluation.
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=validation_split,
)
test_datagen = ImageDataGenerator(rescale=1.0 / 255)
train_generator = train_datagen.flow_from_directory(
    train_dir,
    classes=train_classes,
    target_size=image_size,
    batch_size=batch_size,
    class_mode="binary",
    subset="training",
    seed=random_seed,
)
validation_generator = train_datagen.flow_from_directory(
    train_dir,
    classes=train_classes,
    target_size=image_size,
    batch_size=batch_size,
    class_mode="binary",
    subset="validation",
    shuffle=False,
    seed=random_seed,
)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    classes=test_classes,
    target_size=image_size,
    batch_size=batch_size,
    class_mode="binary",
    shuffle=False,
)
if train_generator.class_indices != validation_generator.class_indices:
    raise ValueError("Training and validation class mappings do not match.")
if train_generator.class_indices != test_generator.class_indices:
    raise ValueError("Training and test class mappings do not match.")
print("Class-to-index mapping:", train_generator.class_indices)


In [ ]:
model = Sequential(
    [
        Input(shape=(*image_size, 3)),
        Conv2D(32, (3, 3), activation="relu"),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation="relu"),
        MaxPooling2D((2, 2)),
        Conv2D(128, (3, 3), activation="relu"),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(512, activation="relu"),
        Dropout(0.5),
        Dense(1, activation="sigmoid"),
    ]
)
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)


In [ ]:
history = model.fit(
    train_generator,
    epochs=20,
    validation_data=validation_generator,
)

# Evaluate the held-out test directory once, after model selection/training.
test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)
test_generator.reset()
test_probabilities = model.predict(test_generator, verbose=0).ravel()
test_predictions = (test_probabilities >= 0.5).astype(int)
class_names = [
    label for label, _ in sorted(train_generator.class_indices.items(), key=lambda item: item[1])
]
print(f"Held-out test loss: {test_loss:.4f}")
print(f"Held-out test accuracy: {test_accuracy:.4f}")
print("Confusion matrix:\n", confusion_matrix(test_generator.classes, test_predictions))
print(
    "Classification report:\n",
    classification_report(
        test_generator.classes,
        test_predictions,
        labels=[0, 1],
        target_names=class_names,
        zero_division=0,
    ),
)
if len(np.unique(test_generator.classes)) == 2:
    print(f"Held-out test ROC-AUC: {roc_auc_score(test_generator.classes, test_probabilities):.4f}")
else:
    print("ROC-AUC skipped because the test set has one observed class.")


In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs = range(1, len(acc) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, acc, label="Train")
axes[0].plot(epochs, val_acc, label="Validation")
axes[0].set(title="Accuracy", xlabel="Epoch")
axes[0].legend()
axes[1].plot(epochs, loss, label="Train")
axes[1].plot(epochs, val_loss, label="Validation")
axes[1].set(title="Loss", xlabel="Epoch")
axes[1].legend()
fig.tight_layout()
plt.show()


In [ ]:
from tensorflow.keras.preprocessing.image import img_to_array, load_img

def predict_image(model, img_path, class_indices):
    """Print a dataset-label prediction; this is not a medical diagnosis."""
    image_path = Path(img_path).expanduser()
    if not image_path.is_file():
        raise FileNotFoundError(f"Prediction image is missing: {image_path}")
    image = load_img(image_path, target_size=image_size)
    image_array = np.expand_dims(img_to_array(image) / 255.0, axis=0)
    score_class_1 = float(model.predict(image_array, verbose=0)[0][0])
    labels_by_index = {index: label for label, index in class_indices.items()}
    if set(labels_by_index) != {0, 1}:
        raise ValueError(f"Expected binary class indices 0 and 1; found {class_indices}.")
    predicted_index = int(score_class_1 >= 0.5)
    print(f"Model-predicted dataset label: {labels_by_index[predicted_index]}")
    print(f"Score for class 1 ({labels_by_index[1]}): {score_class_1:.3f}")

prediction_image = os.environ.get("PREDICT_IMAGE")
if prediction_image:
    predict_image(model, prediction_image, train_generator.class_indices)


In [ ]:
model_path.parent.mkdir(parents=True, exist_ok=True)
model.save(model_path)
if not model_path.is_file():
    raise RuntimeError(f"Model artifact was not created at {model_path}")
# A successful save is not enough: verify that a fresh process can deserialize the artifact.
try:
    loaded_model = tf.keras.models.load_model(model_path, compile=False)
except (OSError, ValueError, TypeError) as exc:
    raise RuntimeError(f"Saved model artifact could not be loaded from {model_path}") from exc
if tuple(loaded_model.input_shape[1:]) != (*image_size, 3) or tuple(loaded_model.output_shape[1:]) != (1,):
    raise RuntimeError(f"Saved model artifact has an unexpected shape: {loaded_model.input_shape} -> {loaded_model.output_shape}")
print(f"Saved and reloaded model artifact: {model_path}")
